# ChessLab: Streaming Dataset Generator on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mochiron-desu/chessAI/blob/master/colab/generate_data.ipynb)

This notebook streams standard rated games directly from **Lichess open databases**, filters for high quality, deduplicates positions, writes packed memory-mapped binary shards (`train.bin`, `val.bin`, `test.bin`), and **packages them into a downloadable `.zip` archive**.

### Features:
- **No Google Drive Required**: Runs entirely in Colab's local disk space (`/content`).
- **Zero-Disk-Overflow Streaming**: Decompresses `.pgn.zst` on the fly over HTTP without saving raw multi-gigabyte PGN files.
- **Scalable Up to 10M–100M Positions**: Configurable via Colab Form widgets.
- **Automatic Zip Packaging & Browser Download**: One-click download of the dataset for use in `train_gpu.ipynb`.

## 1. Auto-Clone Repository & Install Requirements

In [ ]:
# =============================================================================
# STEP 1: Auto-Clone Repo & Configure chesslab
# =============================================================================
import os
import sys

# Auto-clone repository if running in Colab and files not yet present
if not os.path.exists("src/chesslab"):
    if not os.path.exists("/content/chessAI"):
        print("Cloning https://github.com/mochiron-desu/chessAI.git...")
        !git clone https://github.com/mochiron-desu/chessAI.git /content/chessAI
    if os.path.exists("/content/chessAI"):
        os.chdir("/content/chessAI")
        print(f"Switched directory to: {os.getcwd()}")

# Install dependencies & local package
!pip install -q python-chess zstandard pyyaml tqdm requests scipy
if os.path.exists("pyproject.toml"):
    !pip install -q -e .

for path in [os.path.abspath("src"), "/content/chessAI/src", "/content/src"]:
    if os.path.exists(path) and path not in sys.path:
        sys.path.insert(0, path)

import chesslab
print(f"SUCCESS: chesslab ready from {chesslab.__file__}")

## 2. Interactive Dataset Configuration Form

Configure target size, Lichess archive source, quality filters, and download settings.

In [ ]:
#@title Dataset Configuration { run: "auto" }

# Target number of valid games to scan and extract
TARGET_GAMES = 100000 #@param [10000, 25000, 50000, 100000, 500000, 1000000, 3000000, 10000000] {type:"raw", allow-input: true}

# Number of positions sampled per valid game (use 1 for standard, or up to 10 to scale to 10M-100M positions)
POSITIONS_PER_GAME = 1 #@param [1, 2, 3, 5, 10] {type:"raw", allow-input: true}

# Lichess standard rated monthly archive (or specify custom URL)
LICHESS_MONTH = "2024-01" #@param ["2024-01", "2024-02", "2023-12", "2023-11", "2023-10", "2023-09", "2023-08", "2023-07", "2023-06", "2023-05", "2023-04", "2023-03", "2023-02", "2023-01"] {allow-input: true}

# Quality filters
MIN_ELO = 1500 #@param {type:"integer"}
MIN_PLY = 12 #@param {type:"integer"}

# Output and reporting
OUTPUT_DIR = "data" #@param {type:"string"}
REPORT_INTERVAL = 10000 #@param {type:"integer"}
AUTO_ZIP_AND_DOWNLOAD = True #@param {type:"boolean"}

source_url = f"https://database.lichess.org/standard/lichess_db_standard_rated_{LICHESS_MONTH}.pgn.zst"
expected_positions = TARGET_GAMES * POSITIONS_PER_GAME
expected_shard_mb = (expected_positions * 348) / (1024 * 1024)

print("=== Planned Configuration ===")
print(f"  Source URL            : {source_url}")
print(f"  Target Valid Games    : {TARGET_GAMES:,}")
print(f"  Positions Per Game    : {POSITIONS_PER_GAME}")
print(f"  Expected Positions    : {expected_positions:,}")
print(f"  Expected Shard Size   : ~{expected_shard_mb:.1f} MB (packed 348 bytes/record)")
print(f"  Min Elo Threshold     : {MIN_ELO}")
print(f"  Auto Zip & Download   : {AUTO_ZIP_AND_DOWNLOAD}")

## 3. Streaming Ingestion Pipeline Execution

Runs streaming decompression and binary serialization with zero disk overflow.

In [ ]:
import io
import os
import time
import json
import random
import hashlib
import urllib.request
from pathlib import Path
import zstandard as zstd
import chess
import chess.pgn
import numpy as np

from chesslab.data.stream_parser import RECORD_DTYPE, encode_position_to_record, filter_game
from chesslab.data.pipeline import assign_split_hash

shards_dir = os.path.join(OUTPUT_DIR, "shards")
manifests_dir = os.path.join(OUTPUT_DIR, "manifests")
os.makedirs(shards_dir, exist_ok=True)
os.makedirs(manifests_dir, exist_ok=True)

writers = {
    "train": open(os.path.join(shards_dir, "train.bin"), "wb"),
    "val": open(os.path.join(shards_dir, "val.bin"), "wb"),
    "test": open(os.path.join(shards_dir, "test.bin"), "wb"),
}

counts = {"train": 0, "val": 0, "test": 0}
seen_hashes = set()
total_scanned = 0
filtered_out = 0
duplicates = 0
valid_games = 0
total_positions = 0

t0 = time.time()
print(f"Opening streaming connection to: {source_url}...")
req = urllib.request.Request(source_url, headers={"User-Agent": "ChessLab-Research-Bot/1.0"})
raw_stream = urllib.request.urlopen(req)

dctx = zstd.ZstdDecompressor()
stream_reader = dctx.stream_reader(raw_stream)
text_stream = io.TextIOWrapper(stream_reader, encoding="utf-8", errors="ignore")

print(f"Streaming started. Target valid games: {TARGET_GAMES:,} ({expected_positions:,} positions)...")

def sample_multiple_positions(game, num_positions, min_ply):
    moves = list(game.mainline_moves())
    total_plies = len(moves)
    if total_plies < min_ply:
        return []
    if num_positions <= 1:
        target_plies = [random.randint(10, total_plies - 2)]
    else:
        avail = list(range(10, total_plies - 1))
        k = min(len(avail), num_positions)
        target_plies = sorted(random.sample(avail, k))
    
    samples = []
    board = game.board()
    target_set = set(target_plies)
    res_str = game.headers.get("Result", "*")
    for ply_idx, move in enumerate(moves):
        if ply_idx in target_set:
            samples.append((board.copy(), move, res_str))
        board.push(move)
    return samples

try:
    while valid_games < TARGET_GAMES:
        game = chess.pgn.read_game(text_stream)
        if game is None:
            print("Stream finished.")
            break

        total_scanned += 1
        if not filter_game(game.headers, min_elo=MIN_ELO):
            filtered_out += 1
            continue

        # Move hash deduplication
        move_str = " ".join([m.uci() for m in game.mainline_moves()])
        move_hash = hashlib.sha256(move_str.encode()).hexdigest()[:16]
        if move_hash in seen_hashes:
            duplicates += 1
            continue
        seen_hashes.add(move_hash)

        samples = sample_multiple_positions(game, POSITIONS_PER_GAME, MIN_PLY)
        if not samples:
            filtered_out += 1
            continue

        split = assign_split_hash(move_hash, salt="chesslab_2026")
        for b_sample, m_sample, r_str in samples:
            try:
                rec = encode_position_to_record(b_sample, m_sample, r_str)
                writers[split].write(rec.tobytes())
                counts[split] += 1
                total_positions += 1
            except Exception:
                continue

        valid_games += 1

        if valid_games % REPORT_INTERVAL == 0 or valid_games == TARGET_GAMES:
            elapsed = time.time() - t0
            speed_games = valid_games / max(0.1, elapsed)
            speed_pos = total_positions / max(0.1, elapsed)
            print(
                f"Valid Games: {valid_games:,}/{TARGET_GAMES:,} ({valid_games/TARGET_GAMES*100:.1f}%) | "
                f"Positions: {total_positions:,} | "
                f"Scanned: {total_scanned:,} | "
                f"Speed: {speed_games:.0f} games/s ({speed_pos:.0f} pos/s) | "
                f"Elapsed: {elapsed:.1f}s"
            )

finally:
    for w in writers.values():
        w.close()
    raw_stream.close()

total_time = time.time() - t0
print(f"\nIngestion complete in {total_time:.1f}s!")
print(f"Final Counts by Split:")
print(f"  Train : {counts['train']:,} positions ({os.path.getsize(os.path.join(shards_dir, 'train.bin')) / (1024*1024):.2f} MB)")
print(f"  Val   : {counts['val']:,} positions ({os.path.getsize(os.path.join(shards_dir, 'val.bin')) / (1024*1024):.2f} MB)")
print(f"  Test  : {counts['test']:,} positions ({os.path.getsize(os.path.join(shards_dir, 'test.bin')) / (1024*1024):.2f} MB)")

## 4. Manifest Generation & Integrity Check

In [ ]:
# Compute SHA-256 shard checksums and write manifest
shard_hashes = {}
shard_sizes = {}
for split in ["train", "val", "test"]:
    p = os.path.join(shards_dir, f"{split}.bin")
    shard_sizes[split] = os.path.getsize(p)
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    shard_hashes[split] = h.hexdigest()

manifest = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "config": {
        "source": source_url,
        "output_dir": OUTPUT_DIR,
        "target_games": TARGET_GAMES,
        "positions_per_game": POSITIONS_PER_GAME,
        "min_elo": MIN_ELO,
        "min_ply": MIN_PLY,
    },
    "total_scanned": total_scanned,
    "filtered_out": filtered_out,
    "duplicates": duplicates,
    "valid_games": valid_games,
    "counts_by_split": counts,
    "shard_sizes_bytes": shard_sizes,
    "shard_sha256": shard_hashes,
    "total_time_seconds": round(total_time, 2),
    "positions_per_second": round(total_positions / max(0.1, total_time), 2),
}

manifest_path = os.path.join(manifests_dir, "dataset_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest saved to: {manifest_path}")
print(json.dumps(manifest, indent=2))

## 5. Zip Packaging & Automatic Browser Download

Compresses binary shards and manifest into a single `.zip` file and triggers browser download.

In [ ]:
import zipfile

zip_filename = f"chesslab_dataset_{total_positions}_positions.zip"
print(f"Packing binary shards and manifest into: {zip_filename}...")

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [shards_dir, manifests_dir]:
        for root, _, files in os.walk(folder):
            for file in files:
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, start=".")
                zf.write(full_path, arcname=rel_path)

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"\nZip archive created successfully: {zip_filename} ({zip_size_mb:.2f} MB)")
print("Contents:")
with zipfile.ZipFile(zip_filename, "r") as zf:
    for info in zf.infolist():
        print(f"  - {info.filename:40s} ({info.file_size / (1024*1024):6.2f} MB)")

if AUTO_ZIP_AND_DOWNLOAD:
    try:
        from google.colab import files
        print("\nTriggering browser download for Google Colab...")
        files.download(zip_filename)
    except ImportError:
        print(f"\nRunning in local environment; archive available at: {os.path.abspath(zip_filename)}")